# MarkProbe Core: COCO train2014 end-to-end run

Prepares COCO and trains the MarkProbe-Core PyTorch network with the crop+camera message branch enabled by default and fixed step-based loss ramps. Held-out attack-matrix and gradient/loss diagnostics are retained; validation-gated image-loss scheduling remains available through the API/CLI. Set `FAST_SMOKE_TEST=True` only for a short pipeline check.


In [ ]:
from pathlib import Path
import sys
import zipfile
import math
import numpy as np
import torch
from PIL import Image, ImageDraw
from IPython.display import display, Markdown
from tqdm import tqdm
import torch.nn.functional as F

PROJECT = Path.cwd().resolve()
if not (PROJECT / 'markprobe_core').is_dir():
    PROJECT = PROJECT.parent
if not (PROJECT / 'markprobe_core').is_dir():
    raise FileNotFoundError('Cannot find markprobe_core; open this notebook from the MarkProbe-Core project.')
sys.path.insert(0, str(PROJECT))
from markprobe_core import train_model, load_checkpoint, encode_image, decode_image, text_to_bits, bits_to_text

FAST_SMOKE_TEST = False  # Set True only for a short pipeline smoke test; it is not quality training.

DEVICE = torch.device('cuda:0' if torch.cuda.is_available() else 'cpu')
if DEVICE.type == 'cuda':
    print(f'GPU training/inference device: {torch.cuda.get_device_name(DEVICE)} ({DEVICE})')
else:
    print('CUDA is not available; using CPU. Use a CUDA-enabled PyTorch environment for practical full-dataset training.')

ZIP_PATH = Path('/public/data/image/COCO2014/train2014.zip')
DATASET_DIR = Path('/workspace/dataset/coco2014')
OUTPUT_DIR = PROJECT / 'runs'
EXTRACT_MARKER = DATASET_DIR / '.train2014_extracted'
IMAGE_EXTENSIONS = {'.jpg', '.jpeg', '.png', '.bmp', '.webp'}

display(Markdown('## Stage 1/4: prepare the COCO train2014 dataset'))
print(f'Dataset archive: {ZIP_PATH}')
print(f'Dataset extraction/training directory: {DATASET_DIR}')
if not EXTRACT_MARKER.exists():
    if not ZIP_PATH.is_file():
        raise FileNotFoundError(f'COCO archive not found: {ZIP_PATH}')
    DATASET_DIR.mkdir(parents=True, exist_ok=True)
    with zipfile.ZipFile(ZIP_PATH) as archive:
        files = [item for item in archive.infolist() if not item.is_dir()]
        with tqdm(files, desc='Extract train2014.zip', unit='file', mininterval=1.0) as progress:
            for item in progress:
                archive.extract(item, DATASET_DIR)
    EXTRACT_MARKER.touch()
else:
    print(f'Extraction marker found; reusing {DATASET_DIR}')

image_paths = sorted(path for path in DATASET_DIR.rglob('*') if path.is_file() and path.suffix.lower() in IMAGE_EXTENSIONS)
if not image_paths:
    raise RuntimeError(f'No supported images found under {DATASET_DIR}')
VAL_FRACTION = 0.1  # Held-out split is used for scenario metrics and checkpoint selection.
VAL_COUNT = max(1, int(len(image_paths) * VAL_FRACTION)) if VAL_FRACTION > 0 and len(image_paths) > 1 else 0
print('Dataset loaded successfully')
print(f'  Resolved path: {DATASET_DIR.resolve()}')
print(f'  Total images: {len(image_paths):,}')
print(f'  Training images: {len(image_paths) - VAL_COUNT:,} | Validation images: {VAL_COUNT:,}')
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

IMAGE_SIZE = 400  # StegaStamp reference resolution; do not downscale for canonical training.
MESSAGE_BITS = 96  # Shared byte-aligned default across notebook, CLI, API, and text payloads.
WIDTH = 32
EPOCHS = 1 if FAST_SMOKE_TEST else 1000
MAX_TRAIN_STEPS = None
BATCH_SIZE = 4  # Matches the original StegaStamp training default.
USE_STN = True
RESIDUAL_SCALE = 0.1  # Metadata compatibility only; the reference residual is linear and unscaled.
CHECKPOINT = OUTPUT_DIR / 'markprobe_core.pt'
LATEST_CHECKPOINT = OUTPUT_DIR / 'markprobe_core.last.pt'
TRAIN_IMAGE_LIMIT = 512 if FAST_SMOKE_TEST else None  # Smoke subset or full COCO train split.
DUAL_BRANCH = True  # Train both the regular camera path and crop+camera path.
ADAPTIVE_IMAGE_LOSS = False  # StegaStamp reference uses fixed step-based loss ramps.
ADAPTIVE_MSG_THRESHOLD = 1.0 if FAST_SMOKE_TEST else 0.30
ADAPTIVE_BER_THRESHOLD = 0.50 if FAST_SMOKE_TEST else 0.25
ADAPTIVE_PATIENCE = 2
ADAPTIVE_RAMP_STEPS = 8 if FAST_SMOKE_TEST else 15000
VALIDATION_SAMPLES = 32 if FAST_SMOKE_TEST else 256
GRADIENT_LOG_INTERVAL = 1 if FAST_SMOKE_TEST else 100
METRICS_LOG = OUTPUT_DIR / 'markprobe_core.metrics.jsonl'
NO_IM_LOSS_STEPS = 5 if FAST_SMOKE_TEST else 1500
LOSS_RAMP_STEPS = 50 if FAST_SMOKE_TEST else 15000
GEOMETRY_RAMP_STEPS = 50 if FAST_SMOKE_TEST else 10000
JPEG_RAMP_STEPS = 50 if FAST_SMOKE_TEST else 1000
EDGE_DELAY_STEPS = 100 if FAST_SMOKE_TEST else 80000
EDGE_RAMP_STEPS = 50 if FAST_SMOKE_TEST else 10000
EXPECTED_TRAIN_CONFIG = {
    'batch_size': BATCH_SIZE,
    'lr': 1e-4,
    'seed': 2026,
    'val_fraction': VAL_FRACTION,
    'no_im_loss_steps': NO_IM_LOSS_STEPS,
    'l2_loss_scale': 2.0,
    'l2_loss_ramp_steps': LOSS_RAMP_STEPS,
    'lpips_loss_ramp_steps': LOSS_RAMP_STEPS,
    'geometry_ramp_steps': GEOMETRY_RAMP_STEPS,
    'jpeg_ramp_steps': JPEG_RAMP_STEPS,
    'lpips_weight': 1.5,
    'secret_loss_scale': 1.5,
    'secret_loss_ramp_steps': 1,
    'yuv_scales': (1.0, 100.0, 100.0),
    'edge_gain': 10.0,
    'edge_delay_steps': EDGE_DELAY_STEPS,
    'edge_ramp_steps': EDGE_RAMP_STEPS,
    'gan_weight': 0.5,
    'gan_ramp_steps': LOSS_RAMP_STEPS,
    'use_gan': True,
    'use_jpeg': True,
    'border': 'white',
    'dual_branch': DUAL_BRANCH,
    'crop_scale_min': 0.65,
    'crop_scale_max': 0.90,
    'adaptive_image_loss': ADAPTIVE_IMAGE_LOSS,
    'adaptive_msg_threshold': ADAPTIVE_MSG_THRESHOLD,
    'adaptive_ber_threshold': ADAPTIVE_BER_THRESHOLD,
    'adaptive_patience': ADAPTIVE_PATIENCE,
    'adaptive_ramp_steps': ADAPTIVE_RAMP_STEPS,
    'validation_samples': VALIDATION_SAMPLES,
}
RESUME_FROM = None
if LATEST_CHECKPOINT.is_file():
    saved_state = torch.load(LATEST_CHECKPOINT, map_location='cpu', weights_only=False)
    expected_arch = (MESSAGE_BITS, WIDTH, USE_STN, RESIDUAL_SCALE, IMAGE_SIZE)
    saved_arch = (saved_state.get('message_bits'), saved_state.get('width'),
                  saved_state.get('use_stn'), saved_state.get('residual_scale'),
              saved_state.get('image_size'))
    saved_config = saved_state.get('training_state', {}).get('train_config', {})
    if (saved_state.get('format') == 'markprobe-core-v6-robust-training'
            and saved_arch == expected_arch and saved_config == EXPECTED_TRAIN_CONFIG):
        RESUME_FROM = LATEST_CHECKPOINT
    else:
        print('Found an old/incompatible training checkpoint; it will not be resumed. The network architecture is unchanged, but the v6 objective/optimizer state differs, so start a new run (legacy weights remain inference-loadable).')
print(f'Training config: image_size={IMAGE_SIZE}, width={WIDTH}, message_bits={MESSAGE_BITS}, STN={USE_STN}, epochs={EPOCHS}, max_steps={MAX_TRAIN_STEPS}, batch_size={BATCH_SIZE}')
print(f'Best checkpoint path: {CHECKPOINT}')
print(f'Training image limit: {TRAIN_IMAGE_LIMIT if TRAIN_IMAGE_LIMIT else "all images"}')
print(f'Validation matrix: {VALIDATION_SAMPLES} held-out images | diagnostics: {METRICS_LOG}')
print(f'Resume checkpoint: {RESUME_FROM if RESUME_FROM else "none (start fresh)"}')


## Stage 2/4: train the model

The architecture is unchanged. Training follows the StegaStamp reference preset: one transformed-image message branch, a 1,500-step secret-only warm-up, then linearly ramped YUV/LPIPS/GAN objectives and perspective strength. The 10% held-out split is diagnostic only; it does not gate training losses. Validation still evaluates clean PNG, photo, crop, crop+photo and JPEG75, and logs per-loss, image-quality, and encoder/decoder gradient diagnostics to JSONL. Full mode targets 140,000 optimizer steps.


In [ ]:
display(Markdown('### Training: dataset, epochs, batches and rolling loss are shown below'))
model = train_model(
    DATASET_DIR, CHECKPOINT,
    image_size=IMAGE_SIZE, message_bits=MESSAGE_BITS, width=WIDTH,
    epochs=EPOCHS, max_steps=MAX_TRAIN_STEPS, batch_size=BATCH_SIZE, limit=TRAIN_IMAGE_LIMIT, val_fraction=VAL_FRACTION, device=DEVICE,
    resume=RESUME_FROM, use_stn=USE_STN, residual_scale=RESIDUAL_SCALE,
    no_im_loss_steps=NO_IM_LOSS_STEPS, l2_loss_ramp_steps=LOSS_RAMP_STEPS,
    image_weight=2.0, lpips_loss_ramp_steps=LOSS_RAMP_STEPS, geometry_ramp_steps=GEOMETRY_RAMP_STEPS,
    jpeg_ramp_steps=JPEG_RAMP_STEPS, edge_delay_steps=EDGE_DELAY_STEPS,
    edge_ramp_steps=EDGE_RAMP_STEPS, lpips_weight=1.5,
    yuv_scales=(1.0, 100.0, 100.0), secret_loss_scale=1.5, secret_loss_ramp_steps=1,
    gan_weight=0.5, gan_ramp_steps=LOSS_RAMP_STEPS, use_gan=True,
    dual_branch=DUAL_BRANCH, crop_scale_min=0.65, crop_scale_max=0.90,
    adaptive_image_loss=ADAPTIVE_IMAGE_LOSS,
    adaptive_msg_threshold=ADAPTIVE_MSG_THRESHOLD,
    adaptive_ber_threshold=ADAPTIVE_BER_THRESHOLD,
    adaptive_patience=ADAPTIVE_PATIENCE, adaptive_ramp_steps=ADAPTIVE_RAMP_STEPS,
    validation_samples=VALIDATION_SAMPLES, gradient_log_interval=GRADIENT_LOG_INTERVAL,
    metrics_path=METRICS_LOG,
)
print(f'Training complete; best checkpoint: {CHECKPOINT}')
print(f'Per-epoch validation and gradient diagnostics: {METRICS_LOG}')


## Stage 3/4: load the best compatible checkpoint

The notebook consistently uses `CHECKPOINT`. Legacy v3-v5 weights remain loadable for inference, but only a v6 checkpoint with matching architecture and training configuration is resumable.


In [ ]:
model, device = load_checkpoint(CHECKPOINT, device=DEVICE)
cover_path = image_paths[0]
print(f'Model loaded | device={device} | image={cover_path.name}')


## Stage 4/4: encode, decode and evaluate quality


In [ ]:
text = 'demo'
payload = text_to_bits(text, MESSAGE_BITS)
cover = Image.open(cover_path).convert('RGB').resize((IMAGE_SIZE, IMAGE_SIZE), Image.Resampling.BICUBIC)
cover_tensor = torch.from_numpy(np.asarray(cover, dtype=np.float32) / 255.0).permute(2, 0, 1).unsqueeze(0).to(device)
message_tensor = torch.as_tensor(payload, dtype=torch.float32, device=device).reshape(1, -1)
with torch.no_grad():
    float_encoded = model.encode(cover_tensor, message_tensor).clamp(0.0, 1.0)
    float_delta = (float_encoded - cover_tensor).abs()
    float_mse = float((float_encoded - cover_tensor).square().mean().item())
    float_delta_max = float(float_delta.max().item())
    float_delta_mean = float(float_delta.mean().item())

with tqdm(total=4, desc='Encode/decode evaluation', unit='stage') as progress:
    marked = encode_image(model, cover, payload, device)
    progress.set_postfix_str('encoded image created')
    progress.update(1)
    encoded_path = OUTPUT_DIR / 'encoded.png'
    marked.save(encoded_path)
    progress.set_postfix_str('PNG saved (8-bit quantization)')
    progress.update(1)
    decoded_bits, probabilities = decode_image(model, encoded_path, device)
    progress.set_postfix_str('PNG decoded')
    progress.update(1)
    jpeg_path = OUTPUT_DIR / 'encoded_q75.jpg'
    marked.save(jpeg_path, quality=75, subsampling=2)
    jpeg_bits, jpeg_probabilities = decode_image(model, jpeg_path, device)
    progress.set_postfix_str('JPEG quality-75 decoded')
    progress.update(1)

correct_bits = sum(expected == actual for expected, actual in zip(payload, decoded_bits))
bit_accuracy = correct_bits / MESSAGE_BITS
bit_error_rate = 1.0 - bit_accuracy
jpeg_correct_bits = sum(expected == actual for expected, actual in zip(payload, jpeg_bits))
jpeg_bit_accuracy = jpeg_correct_bits / MESSAGE_BITS

cover_array = np.asarray(cover, dtype=np.float32) / 255.0
marked_array = np.asarray(Image.open(encoded_path).convert('RGB'), dtype=np.float32) / 255.0
if cover_array.shape != marked_array.shape:
    raise ValueError(f'Cover and encoded image sizes differ: {cover_array.shape} vs {marked_array.shape}')
mse = float(np.mean(np.square(cover_array - marked_array)))
psnr = float('inf') if mse == 0 else 10.0 * math.log10(1.0 / mse)
changed_pixel_ratio = float(np.any(cover_array != marked_array, axis=2).mean())
quantized_delta_max = float(np.abs(cover_array - marked_array).max())
cover_tensor_metric = torch.from_numpy(cover_array).permute(2, 0, 1).unsqueeze(0).to(device)
marked_tensor_metric = torch.from_numpy(marked_array).permute(2, 0, 1).unsqueeze(0).to(device)
ssim_window = min(11, cover.height, cover.width)
if ssim_window % 2 == 0:
    ssim_window -= 1
if ssim_window < 3:
    raise ValueError('SSIM evaluation requires image dimensions of at least 3x3')
with torch.no_grad():
    mu_cover = F.avg_pool2d(cover_tensor_metric, ssim_window, stride=1)
    mu_marked = F.avg_pool2d(marked_tensor_metric, ssim_window, stride=1)
    var_cover = (F.avg_pool2d(cover_tensor_metric.square(), ssim_window, stride=1) - mu_cover.square()).clamp_min(0)
    var_marked = (F.avg_pool2d(marked_tensor_metric.square(), ssim_window, stride=1) - mu_marked.square()).clamp_min(0)
    covariance = F.avg_pool2d(cover_tensor_metric * marked_tensor_metric, ssim_window, stride=1) - mu_cover * mu_marked
    c1, c2 = 0.01 ** 2, 0.03 ** 2
    ssim = float((((2 * mu_cover * mu_marked + c1) * (2 * covariance + c2)) /
                  ((mu_cover.square() + mu_marked.square() + c1) * (var_cover + var_marked + c2))).mean().item())
try:
    decoded_text = bits_to_text(decoded_bits)
    text_status = decoded_text
except (ValueError, UnicodeDecodeError):
    decoded_text = None
    text_status = 'Invalid text payload (the model may need more training)'

print(f'Target text: {text}')
print(f'Decoded PNG text: {text_status}')
print(f'Exact text match: {decoded_text == text}')
print(f'PNG bit accuracy: {bit_accuracy:.2%} ({correct_bits}/{MESSAGE_BITS})')
print(f'PNG bit error rate (BER): {bit_error_rate:.2%}')
print(f'PNG mean bit confidence: {float(np.mean([max(p, 1.0 - p) for p in probabilities])):.2%}')
print(f'JPEG quality-75 bit accuracy: {jpeg_bit_accuracy:.2%} ({jpeg_correct_bits}/{MESSAGE_BITS})')
print(f'JPEG mean bit confidence: {float(np.mean([max(p, 1.0 - p) for p in jpeg_probabilities])):.2%}')
print(f'Float encoded MSE before 8-bit save: {float_mse:.8f}')
print(f'Float residual: mean={float_delta_mean:.6f}, max={float_delta_max:.6f}')
print(f'PNG residual: max={quantized_delta_max:.6f}, changed-pixel ratio={changed_pixel_ratio:.2%}')
print(f'PNG image MSE (lower is better): {mse:.8f}')
print(f'PNG image PSNR (dB, higher is better): {psnr:.2f}')
print(f'PNG image SSIM (closer to 1 is better): {ssim:.6f}')
print(f"Decoded bits: {''.join(map(str, decoded_bits))}")
print(f'Encoded PNG: {encoded_path}')
print(f'JPEG robustness sample: {jpeg_path}')
cover_preview = cover.copy()
marked_preview = marked.copy()
cover_preview.thumbnail((512, 512))
marked_preview.thumbnail((512, 512))
label_height = 28
preview = Image.new('RGB', (cover_preview.width + marked_preview.width, max(cover_preview.height, marked_preview.height) + label_height), 'white')
draw = ImageDraw.Draw(preview)
draw.text((8, 7), 'Original cover', fill='black')
draw.text((cover_preview.width + 8, 7), 'Watermarked', fill='black')
preview.paste(cover_preview, (0, label_height))
preview.paste(marked_preview, (cover_preview.width, label_height))
display(preview)


> A single epoch is only a pipeline smoke test. For meaningful recovery, train longer and inspect the JSONL log for held-out attack-matrix and gradient/loss diagnostics. Gate thresholds and ramp lengths are starting points to validate on the target GPU and data domain, not guaranteed convergence criteria.
